In [42]:
import pandas as pd
import numpy as np
import plotly.express as px
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from Forecaster.weather_utils import WeatherUtils

In [43]:
arch_hall_validation_results_path = './../Results/Arch_Hall_ST2B13_Validation_Results.csv'
arch_hall_validation_results = pd.read_csv(arch_hall_validation_results_path, index_col=0, parse_dates=True)

arch_hall_test_results_path = './../Results/Arch_Hall_ST2B13_Test_Results.csv'
arch_hall_test_results = pd.read_csv(arch_hall_test_results_path, index_col=0, parse_dates=True)

arch_hall_validation_results.shape, arch_hall_test_results.shape

((312, 4), (696, 4))

In [44]:
combined_results = pd.concat([arch_hall_validation_results, arch_hall_test_results], axis=0)
combined_results.shape

(1008, 4)

In [45]:
start_date = combined_results.index[0].strftime('%Y-%m-%d')
end_date = combined_results.index[-1].strftime('%Y-%m-%d')

start_date, end_date

('2024-06-18', '2024-07-31')

In [46]:
weather_utils = WeatherUtils()
historic_weather = weather_utils.fetch_historic_data_from_api(13.1765314, -59.6168188, start_date, end_date)
historic_weather

Coordinates 13.181018829345703°N -59.56243896484375°E
Elevation 114.0 m asl
Timezone b'America/Barbados' b'GMT-4'
Timezone difference to GMT+0 -14400 s


,temperature_2m_historic,relative_humidity_2m_historic,rain_historic,snowfall_historic,weather_code_historic,pressure_msl_historic,surface_pressure_historic,wind_speed_10m_historic,wind_direction_10m_historic,shortwave_radiation_historic,cloud_cover_historic,cloud_cover_low_historic,direct_normal_irradiance_historic,diffuse_radiation_historic,sunshine_duration_historic,is_day_historic,shortwave_radiation_instant_historic,direct_normal_irradiance_instant_historic,diffuse_radiation_instant_historic
date,,,,,,,,,,,,,,,,,,,
2024-06-18 00:00:00,25.602501,86.611610,0.0,0.0,3.0,1012.099976,999.007935,14.372974,67.932053,0.0,81.0,24.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
2024-06-18 01:00:00,25.602501,86.873665,0.0,0.0,2.0,1011.700012,998.613159,14.241630,69.274361,0.0,71.0,28.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
2024-06-18 02:00:00,25.552500,87.659668,0.0,0.0,3.0,1011.099976,998.018616,14.417988,87.137657,0.0,84.0,19.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
2024-06-18 03:00:00,25.302500,89.510300,0.0,0.0,3.0,1011.599976,998.501587,13.910169,79.562576,0.0,100.0,31.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
2024-06-18 04:00:00,24.802500,90.016251,0.3,0.0,51.0,1012.000000,998.874451,12.303366,110.556129,0.0,99.0,29.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-07-31 19:00:00,26.552500,81.385765,0.0,0.0,1.0,1015.200012,1002.109131,3.319036,77.471199,6.0,25.0,1.0,12.38072,5.0,0.0,0.0,0.0,0.0,0.0
2024-07-31 20:00:00,26.152500,84.854027,0.0,0.0,0.0,1015.900024,1002.782776,3.617955,84.289497,0.0,12.0,3.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
2024-07-31 21:00:00,26.152500,87.975182,0.0,0.0,0.0,1016.700012,1003.572449,2.880000,360.000000,0.0,5.0,2.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0


In [47]:
def get_errors_from_results(results):
    errors = results.copy()
    errors['ANN_RLS_Error'] = errors.apply(lambda row: np.abs(row['ANN_RLS'] - row['Actual']), axis=1)
    errors['LSTM_RLS_Error'] = errors.apply(lambda row: np.abs(row['LSTM_RLS'] - row['Actual']), axis=1)
    errors['Final_RLS_Error'] = errors.apply(lambda row: np.abs(row['Final_RLS'] - row['Actual']), axis=1)
    
    errors.drop(columns=['ANN_RLS', 'LSTM_RLS', 'Final_RLS', 'Actual'], inplace=True)
    
    return errors

combined_errors = get_errors_from_results(combined_results)
combined_errors_day = combined_errors.resample('D').mean()

combined_errors['Weather_Code'] = historic_weather['weather_code_historic'].astype(int)
combined_errors['Weather_Code'] = combined_errors['Weather_Code'].astype('category')
combined_errors

,ANN_RLS_Error,LSTM_RLS_Error,Final_RLS_Error,Weather_Code
2024-06-18 00:00:00,87.995237,92.539122,144.498854,3
2024-06-18 01:00:00,70.238306,146.495979,0.833581,2
2024-06-18 02:00:00,349.812181,206.467825,304.459291,3
2024-06-18 03:00:00,85.380585,203.567078,542.768922,3
2024-06-18 04:00:00,26.062289,200.644284,311.985892,51
...,...,...,...,...
2024-07-31 19:00:00,0.235681,50.550942,52.114135,1
2024-07-31 20:00:00,295.889871,7.355027,25.278812,0
2024-07-31 21:00:00,96.556031,92.680735,274.332570,0
2024-07-31 22:00:00,473.088321,39.170453,199.829158,0


In [48]:
px.line(combined_errors.drop(columns='Weather_Code')).show()


In [49]:
px.histogram(combined_errors, x='ANN_RLS_Error', nbins=30).show()
px.histogram(combined_errors, x='LSTM_RLS_Error', nbins=30).show()
px.histogram(combined_errors, x='Final_RLS_Error', nbins=30).show()

In [50]:
combined_errors['DayOfWeek'] = combined_errors.index.dayofweek
combined_errors['Hour'] = combined_errors.index.hour
week_days = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
combined_errors['DayOfWeek'] = combined_errors['DayOfWeek'].apply(lambda x: week_days[x])

px.box(combined_errors, y='ANN_RLS_Error', x='DayOfWeek').show()
px.box(combined_errors, y='LSTM_RLS_Error', x='DayOfWeek').show()
px.box(combined_errors, y='Final_RLS_Error', x='DayOfWeek').show()

In [51]:
px.box(combined_errors, y='ANN_RLS_Error', x='Hour').show()
px.box(combined_errors, y='LSTM_RLS_Error', x='Hour').show()
px.box(combined_errors, y='Final_RLS_Error', x='Hour').show()

In [52]:
px.box(combined_errors, y='ANN_RLS_Error', x='Weather_Code').show()
px.box(combined_errors, y='LSTM_RLS_Error', x='Weather_Code').show()
px.box(combined_errors, y='Final_RLS_Error', x='Weather_Code').show()

In [59]:
# px.line(combined_errors_day).show()

combined_errors_day['DayOfWeek'] = combined_errors_day.index.dayofweek
week_days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
combined_errors_day['DayOfWeek'] = combined_errors_day['DayOfWeek'].apply(lambda x: week_days[x])
combined_errors_day

,ANN_RLS_Error,LSTM_RLS_Error,Final_RLS_Error,DayOfWeek
2024-06-18,819.801052,834.335986,777.060558,Tuesday
2024-06-19,427.815376,455.406317,426.466570,Wednesday
2024-06-20,697.947286,551.640352,626.574571,Thursday
2024-06-21,594.127197,330.113718,506.125183,Friday
2024-06-22,961.323262,865.111082,1057.390447,Saturday
2024-06-23,860.095720,673.394979,841.702531,Sunday
2024-06-24,817.086172,769.871495,856.159692,Monday
2024-06-25,586.690574,388.807531,510.834798,Tuesday
2024-06-26,437.679336,440.249207,368.689697,Wednesday
2024-06-27,539.081227,503.797963,588.685873,Thursday


In [60]:
px.box(combined_errors_day, y='ANN_RLS_Error', x='DayOfWeek').show()
px.box(combined_errors_day, y='LSTM_RLS_Error', x='DayOfWeek').show()
px.box(combined_errors_day, y='Final_RLS_Error', x='DayOfWeek').show()

In [69]:
day_of_week_names = {
    0: 'Monday',
    1: 'Tuesday',
    2: 'Wednesday',
    3: 'Thursday',
    4: 'Friday',
    5: 'Saturday',
    6: 'Sunday'
}

combined_results['DayOfWeek'] = combined_results.index.dayofweek
combined_results['DayOfWeek'] = combined_results['DayOfWeek'].map(day_of_week_names)
combined_results['Weather_Code'] = historic_weather['weather_code_historic'].astype('category')
combined_results['Month'] = combined_results.index.month

px.box(combined_results, y='Actual', x=combined_results['DayOfWeek']).show()
px.box(combined_results, y='ANN_RLS', x=combined_results['DayOfWeek']).show()
px.box(combined_results, y='LSTM_RLS', x=combined_results['DayOfWeek']).show()
px.box(combined_results, y='Final_RLS', x=combined_results['DayOfWeek']).show()

In [70]:
px.box(combined_results, y='Actual', x=combined_results['Weather_Code']).show()
px.box(combined_results, y='ANN_RLS', x=combined_results['Weather_Code']).show()
px.box(combined_results, y='LSTM_RLS', x=combined_results['Weather_Code']).show()
px.box(combined_results, y='Final_RLS', x=combined_results['Weather_Code']).show()

In [71]:
px.box(combined_results, y='Actual', x=combined_results['Month']).show()
px.box(combined_results, y='ANN_RLS', x=combined_results['Month']).show()
px.box(combined_results, y='LSTM_RLS', x=combined_results['Month']).show()
px.box(combined_results, y='Final_RLS', x=combined_results['Month']).show()

In [73]:
combined_results_daytime = combined_results.between_time('06:00', '20:00')
px.box(combined_results_daytime, y='Actual', x=combined_results_daytime['DayOfWeek']).show()
px.box(combined_results_daytime, y='ANN_RLS', x=combined_results_daytime['DayOfWeek']).show()
px.box(combined_results_daytime, y='LSTM_RLS', x=combined_results_daytime['DayOfWeek']).show()
px.box(combined_results_daytime, y='Final_RLS', x=combined_results_daytime['DayOfWeek']).show()

In [74]:
combined_results_daytime = combined_results.between_time('06:00', '20:00')
px.box(combined_results_daytime, y='Actual', x=combined_results_daytime['Weather_Code']).show()
px.box(combined_results_daytime, y='ANN_RLS', x=combined_results_daytime['Weather_Code']).show()
px.box(combined_results_daytime, y='LSTM_RLS', x=combined_results_daytime['Weather_Code']).show()
px.box(combined_results_daytime, y='Final_RLS', x=combined_results_daytime['Weather_Code']).show()

In [84]:
from scipy.stats import iqr


q1 = np.percentile(combined_results_daytime['Actual'], 25)
q3 = np.percentile(combined_results_daytime['Actual'], 75)

upper_bound = q3 + 1.5*iqr(combined_results_daytime['Actual'])
lower_bound = q1 - 1.5*iqr(combined_results_daytime['Actual'])


upper_bound, lower_bound, q1, q3, iqr

(14596.551093675598,
 -8996.007854538693,
 -148.79824895833346,
 5749.341488095239,
 <function scipy.stats._stats_py.iqr(x, axis=None, rng=(25, 75), scale=1.0, nan_policy='propagate', interpolation='linear', keepdims=False)>)

In [83]:
px.box(combined_results_daytime, y='Actual').show()